In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

In [7]:
from aiida import orm
from monty.serialization import loadfn, dumpfn
import numpy as np
from aiida_vasp.workchains import VaspHybridBandsWorkChain
from aiida_vasp.workchains.v2 import VaspBandUpdater, VaspRelaxUpdater
from aiida_grouppathx import GroupPathX
from tqdm import tqdm

In [3]:
# Load the structures to be calculated
dataset = loadfn('binary_mbj_calc_structures_2025_03_19.json')
print(len(dataset))

665


In [4]:
basepath = GroupPathX('hc-binary-mbj')
structpath = basepath['structures']

Deposit the structures

Sort the nodes based on their number of sites

In [5]:
nodes = [x.get_node() for x in structpath]
nodes.sort(key=lambda x: len(x.sites))

In [6]:
nodes[-1].get_ase()

Atoms(symbols='Ru4F24', pbc=True, cell=[[0.0, 0.0, -4.82221923], [0.0, -8.38575766, 0.0], [-9.05755894, 0.0, 0.0]], masses=...)

## Setting up the calculation

In [19]:
def callback(node, label):
    """Generate process builder"""
    #queue_name = 'tyhcnormal'
    queue_name = 'xhhctdnormal'
    code = 'vasp-6.4.2@sugon-xh-v2'
    upd = VaspBandUpdater().apply_preset(structure=node, overrides={
        'ispin': 2,
        'gga': None,  # PBE functional
        'magmom': None,
        'ncore': 8,
        'kpar': 8,
    },
        # code='vasp-6.4.2@sugon-tai',
        code=code,
        label=f'{node.get_formula()} {node.label} MP STRUCT PBE NORELAX')
    upd.set_resources(num_machines=1, tot_num_mpiprocs=64)
    upd.set_options(max_wallclock_seconds=3600 * 48, queue_name=queue_name)
    upd.set_band_settings(band_mode='bradcrack', line_density=20)
    running = upd.submit()
    return running, label

In [20]:
workpath = basepath['pbe_bandstructure_works']
workpath.get_or_create_group()

(<Group: 'hc-binary-mbj/pbe_bandstructure_works' [type core], of user bzhu@bit.edu.cn>,
 False)

In [21]:
from aiida_grouppathx.launch_manager import GroupLauncher

In [22]:
launcher = GroupLauncher(workpath, 45, callback, source_key_obj_pairs=[(node.label, node) for node in nodes], logfile='launch_2025_4_6.log')

In [23]:
launcher.launch_loop()

Total number of running jobs: 45
Total number of jobs to run : 269
Time elapsed to gather jobs: 0.16 seconds
Slot usage: 45/45
Total number of running jobs: 39
Total number of jobs to run : 269
Time elapsed to gather jobs: 0.21 seconds
Slot usage: 39/45
Launching 6 jobs...
Launched 6 jobs...
Total number of running jobs: 45
Total number of jobs to run : 263
Time elapsed to gather jobs: 0.22 seconds
Slot usage: 45/45
Total number of running jobs: 35
Total number of jobs to run : 263
Time elapsed to gather jobs: 0.18 seconds
Slot usage: 35/45
Launching 10 jobs...
Launched 10 jobs...
Total number of running jobs: 45
Total number of jobs to run : 253
Time elapsed to gather jobs: 0.21 seconds
Slot usage: 45/45
Total number of running jobs: 43
Total number of jobs to run : 253
Time elapsed to gather jobs: 0.23 seconds
Slot usage: 43/45
Launching 2 jobs...
Launched 2 jobs...
Total number of running jobs: 33
Total number of jobs to run : 251
Time elapsed to gather jobs: 0.18 seconds
Slot usage